In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [3]:
from tqdm import tqdm
from pathlib import Path
from datetime import datetime

import numpy as np

In [4]:
import torch
import monai
from monai.utils import ensure_tuple_rep

from src.loader import get_dataloader
from src.utils import load_pretrain_model
from src.utils import same_seeds, load_config
from src.SlimUNETR.SlimUNETR import SlimUNETR
from monai.networks.nets import UNet

from main_unlab import calc_metrics_dict
from main_unlab import get_experiment_dir

from accelerate import Accelerator


In [5]:
device = torch.device('cuda:1')
torch.cuda.set_device(device)

In [6]:
config, data_flag, is_HepaticVessel = load_config()
# config.trainer.batch_size = 4
data_flag

'tbad_dataset'

In [7]:
same_seeds(config.trainer.seed)
image_size = config.trainer.image_size

model = SlimUNETR(**config.slim_unetr)
model.to(device)

train_loader, val_loader, unlab_loader = get_dataloader(config, data_flag)

In [8]:
inference = monai.inferers.SlidingWindowInferer(
    roi_size=ensure_tuple_rep(config.trainer.image_size, dim=3),
    overlap=0.5,
    sw_device=device,
    device=device,
)

metrics = {
    "dice_metric": monai.metrics.DiceMetric(
        include_background=True,
        reduction=monai.utils.MetricReduction.MEAN_BATCH,
        get_not_nans=False,
    ),
    # 'hd95_metric': monai.metrics.HausdorffDistanceMetric(percentile=95, include_background=True, reduction=monai.utils.MetricReduction.MEAN_BATCH, get_not_nans=False)
}

post_trans = monai.transforms.Compose(
    [
        monai.transforms.Activations(sigmoid=True),
        monai.transforms.AsDiscrete(threshold=0.5),
    ]
)


In [9]:
logging_dir = Path(os.getcwd()) / "logs" / str(datetime.now()).replace(":", "_")
accelerator = Accelerator(log_with=["tensorboard"], project_dir=logging_dir)
accelerator.init_trackers("seed test")

In [10]:
base_exp_path_save = get_experiment_dir(config, data_flag, root="model_store")

In [11]:
root_path = Path('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec')

In [12]:
root_str = str(root_path)

In [13]:
root_path

WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec')

In [14]:
def replace_list(seed_list, path):
    for seed in seed_list:
        path = path.replace(seed + '\\', '')
    
    return path

In [15]:
seed_list = list(map(lambda p: p.name, root_path.iterdir()))

exp_list = list(map(lambda p: str(p), root_path.rglob("*/epoch_799")))
exp_list_act = [str(exp) for exp in exp_list]

exp_list = list(map(lambda p: replace_list(seed_list, p), exp_list))
exp_list = list(map(lambda p: p.replace(root_str + '\\', ""), set(exp_list)))

In [16]:
exp_list_with_seed = []
for exp in exp_list:
    exp_list_with_seed.append(list(root_path.rglob("*\\" + exp)))

In [33]:
exp_list_with_seed

[[WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec/seed25/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch1/epoch_799'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec/seed32/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch1/epoch_799'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec/seed42/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_unlab_ratio0.5_unlab_weight0.3_start_unlab_epo

In [17]:
len(['d:', 'sandbox', 'medical', 'Slim-UNETR', 'model_store', 'imageTBAD_MSD_preproc_128_border_crop64', 'tbad_dataset_unlab_stages_with_tflab_selec', 'seed25'])

8

In [18]:
root_str

'd:\\sandbox\\medical\\Slim-UNETR\\model_store\\imageTBAD_MSD_preproc_128_border_crop64\\tbad_dataset_unlab_stages_with_tflab_selec'

In [19]:
# for exp_seed_list in exp_list_with_seed:
#     for exp_seed in exp_seed_list:
#         exp_seed_split = str(exp_seed.parent).split('\\')
#         exp_seed_split.append(exp_seed_split[7])
#         del exp_seed_split[7]

#         exp_seed.parent.rename('\\'.join(exp_seed_split))



#         # print(exp_seed_split[7])
#         # print('\\'.join(exp_seed_split[:7])+'\\'.join(exp_seed_split[7+1:]) )

In [20]:
exp_list_with_seed

[[WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec/seed25/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch1/epoch_799'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec/seed32/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch1/epoch_799'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec/seed42/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_unlab_ratio0.5_unlab_weight0.3_start_unlab_epo

In [21]:

exp_names_list_with_seed = [
    [(exp_seed).parents[-13] for exp_seed in exp_seed_list]
    for exp_seed_list in exp_list_with_seed
]

In [22]:
len(str(exp_list_with_seed[0][0].parents[-13]))

201

In [23]:
exp_names_list_with_seed

[[WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec/seed25/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec/seed32/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec/seed42/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283')],
 [WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_stages_with_tflab_selec/seed25/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/tbad_dataset_unlab_sta

In [29]:
def evaluate_experiment(model, exp_name_path, root_path, device):
    

    metrics_value = []
    for exp_path in tqdm(list(root_path.rglob("*\\" + exp_name_path))):
        checkpoint = str(exp_path / "pytorch_model.bin")
        model = load_pretrain_model(checkpoint, model, verbose=False)
        model.eval()
        
        for image_batch in val_loader:
            logits = inference(image_batch["image"].to(device), model)
            val_outputs = [post_trans(i) for i in logits]
            for metric_name in metrics:
                metrics[metric_name](y_pred=val_outputs, y=image_batch["label"].to(device))

        _, batch_acc = calc_metrics_dict(
        metrics, accelerator, data_flag, is_train=False
        )

        metrics_value.append(batch_acc.cpu().numpy())

    return np.mean(metrics_value, axis=0), np.std(metrics_value, axis=0)

In [30]:
exp_list

['epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\only_labeled_unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch1\\epoch_799',
 'epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\only_labeled_\\epoch_799',
 'epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\ps_post_tf\\unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch200\\epoch_799',
 'epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch1\\epoch_799',
 'epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\ps_post_tf\\unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch100\\epoch_799',
 'epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_

In [31]:
exp_eval_list = {
    exp: evaluate_experiment(model, exp, root_path, device)
    for exp in tqdm(exp_list)
}

100%|██████████| 6/6 [02:14<00:00, 22.44s/it]


In [32]:
exp_eval_list

{'epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\only_labeled_unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch1\\epoch_799': (array([0.82886153, 0.72137564, 0.5806952 ], dtype=float32),
  array([0.00188278, 0.00532269, 0.00415118], dtype=float32)),
 'epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\only_labeled_\\epoch_799': (array([0.86613256, 0.7608071 , 0.608156  ], dtype=float32),
  array([0.00327108, 0.00381468, 0.00547781], dtype=float32)),
 'epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\ps_post_tf\\unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch200\\epoch_799': (array([0.8582674 , 0.77368283, 0.63907117], dtype=float32),
  array([0.00414501, 0.00845161, 0.01796563], dtype=float32)),
 'epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\